# NFL Data Cleaning and Validation

This notebook demonstrates the enhanced pipeline workflow for data cleaning, validation, and model evaluation.

## Overview

The enhanced pipeline provides:
1. **Dataset Validation**: Comprehensive checks for data integrity
2. **Error Recovery**: Automatic solutions for common data issues
3. **Model Evaluation**: Cross-validation with TimeSeriesSplit
4. **Reporting**: Detailed logs and metrics

In [ ]:
import sys
sys.path.insert(0, '../')  # Add backend to path

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Import pipeline components
from enhanced_pipeline import (
    DatasetValidator,
    DatasetMerger,
    ModelEvaluator,
    run_enhanced_pipeline,
    ABBR_FIX
)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. Load and Inspect Dataset

First, let's load the existing dataset and understand its structure.

In [ ]:
# Load the dataset
data_path = Path('../data/Nfl_data_sorted.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nFirst 5 rows:")
df.head()

## 2. Basic Data Quality Checks

In [ ]:
# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Check data types
print("Data types:")
df.dtypes

In [ ]:
# Check unique teams
print("Unique home teams:")
print(sorted(df['home_team'].unique()))
print(f"\nTotal unique home teams: {df['home_team'].nunique()}")

print("\nUnique away teams:")
print(sorted(df['away_team'].unique()))
print(f"\nTotal unique away teams: {df['away_team'].nunique()}")

## 3. Validate Team Codes

Check if any legacy team codes need normalization.

In [ ]:
# Check for legacy codes
print("Team abbreviation mapping (legacy → modern):")
for old, new in ABBR_FIX.items():
    print(f"  {old} → {new}")

# Check if any legacy codes exist in data
all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
legacy_codes = all_teams & set(ABBR_FIX.keys())

if legacy_codes:
    print(f"\n⚠️ WARNING: Legacy codes found: {legacy_codes}")
    print("These should be normalized using ABBR_FIX mapping")
else:
    print("\n✓ All team codes are properly normalized")

## 4. Validate Temporal Ordering

Ensure data is properly sorted chronologically.

In [ ]:
# Create time key and check ordering
df['time_key'] = (df['season'] * 100) + df['week']
is_sorted = df['time_key'].is_monotonic_increasing

print(f"Data is chronologically sorted: {is_sorted}")

if not is_sorted:
    print("\n⚠️ WARNING: Data is not properly sorted")
    print("This could cause issues with time-series validation")
else:
    print("\n✓ Data is properly sorted for time-series analysis")

# Show season/week distribution
print("\nSeason coverage:")
print(df.groupby('season').size())

## 5. Validate Features

Check that all required features for model predictions are present.

In [ ]:
# Expected features for model
BASE_FEATURES = [
    # Home priors
    "home_prior_pf_avg_3", "home_prior_pf_avg_5",
    "home_prior_pa_avg_3", "home_prior_pa_avg_5",
    "home_prior_win_pct_3", "home_prior_win_pct_5",
    # Away priors
    "away_prior_pf_avg_3", "away_prior_pf_avg_5",
    "away_prior_pa_avg_3", "away_prior_pa_avg_5",
    "away_prior_win_pct_3", "away_prior_win_pct_5",
    # Differentials
    "home_minus_away_pf_avg_3", "home_minus_away_pf_avg_5",
    "home_minus_away_pa_avg_3", "home_minus_away_pa_avg_5",
    "home_minus_away_win_pct_3", "home_minus_away_win_pct_5",
]

# Check for missing features
missing_features = set(BASE_FEATURES) - set(df.columns)
extra_features = set(df.columns) - set(BASE_FEATURES)

print(f"Required features present: {len(BASE_FEATURES) - len(missing_features)}/{len(BASE_FEATURES)}")

if missing_features:
    print(f"\n⚠️ Missing features: {missing_features}")
else:
    print("\n✓ All required features are present")

# Check feature distributions
print("\nFeature statistics:")
df[BASE_FEATURES].describe()

## 6. Check for Data Leakage

Verify that rolling features don't include current game data.

In [ ]:
# Check if prior features have NaN values at the start (indicating proper shift)
prior_features = [f for f in BASE_FEATURES if 'prior' in f]

print("Checking for data leakage in prior features...\n")

for col in prior_features[:3]:  # Check first 3 as examples
    first_valid_idx = df[col].first_valid_index()
    first_values = df[col].head(10).tolist()
    
    print(f"{col}:")
    print(f"  First valid index: {first_valid_idx}")
    print(f"  First 10 values: {first_values[:5]}...")
    
    if first_valid_idx == 0:
        print("  ⚠️ WARNING: First value is not NaN - possible leakage")
    else:
        print("  ✓ Proper shift detected")
    print()

## 7. Use Enhanced Pipeline for Comprehensive Validation

Now let's use the enhanced pipeline's validation framework.

In [ ]:
# Create validator
validator = DatasetValidator(df, name="nfl_dataset")

# Run all validations
print("Running comprehensive validations...\n")

schema_valid = validator.validate_schema(BASE_FEATURES + ['season', 'week', 'home_team', 'away_team'])
dtype_valid = validator.validate_datatypes({
    'season': 'int',
    'week': 'int',
    'home_prior_pf_avg_3': 'float',
    'away_prior_pf_avg_3': 'float'
})
team_valid = validator.validate_team_codes(['home_team', 'away_team'])
temporal_valid = validator.validate_temporal_order()
leakage_valid = validator.validate_no_leakage(prior_features)

# Generate and display report
report = validator.generate_report()

print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)
print(f"Schema validation: {'✓ PASS' if schema_valid else '✗ FAIL'}")
print(f"Datatype validation: {'✓ PASS' if dtype_valid else '✗ FAIL'}")
print(f"Team code validation: {'✓ PASS' if team_valid else '✗ FAIL'}")
print(f"Temporal order validation: {'✓ PASS' if temporal_valid else '✗ FAIL'}")
print(f"Leakage validation: {'✓ PASS' if leakage_valid else '✗ FAIL'}")
print("="*60)

all_pass = all([schema_valid, dtype_valid, team_valid, temporal_valid, leakage_valid])
if all_pass:
    print("\n✓ ALL VALIDATIONS PASSED")
else:
    print("\n⚠️ SOME VALIDATIONS FAILED - Check error reports for recovery solutions")

## 8. Sample Data Cleaning Operations

Common data cleaning operations you might need.

In [ ]:
# Example: Normalize team codes
def normalize_team_codes(df, team_columns):
    """Apply ABBR_FIX normalization to team columns."""
    df = df.copy()
    for col in team_columns:
        if col in df.columns:
            df[col] = df[col].replace(ABBR_FIX)
            df[col] = df[col].str.strip().str.upper()
    return df

# Example: Fill missing features with median
def impute_missing_features(df, feature_cols):
    """Fill missing values with median."""
    df = df.copy()
    imputed = {}
    for col in feature_cols:
        if col in df.columns:
            missing = df[col].isna().sum()
            if missing > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                imputed[col] = (missing, median_val)
    return df, imputed

# Example: Ensure proper sorting
def ensure_temporal_order(df):
    """Sort dataframe chronologically."""
    df = df.copy()
    df = df.sort_values(['season', 'week', 'home_game_date']).reset_index(drop=True)
    return df

print("Data cleaning functions defined.")
print("\nAvailable functions:")
print("  - normalize_team_codes(df, team_columns)")
print("  - impute_missing_features(df, feature_cols)")
print("  - ensure_temporal_order(df)")

## 9. Apply Cleaning Operations (If Needed)

In [ ]:
# Create a clean copy
df_clean = df.copy()

# Apply cleaning operations
print("Applying data cleaning operations...\n")

# 1. Normalize team codes
df_clean = normalize_team_codes(df_clean, ['home_team', 'away_team'])
print("✓ Team codes normalized")

# 2. Impute missing features
df_clean, imputed_info = impute_missing_features(df_clean, BASE_FEATURES)
if imputed_info:
    print(f"✓ Imputed {len(imputed_info)} features with missing values")
    for col, (count, median) in imputed_info.items():
        print(f"  {col}: {count} values imputed with median {median:.2f}")
else:
    print("✓ No imputation needed")

# 3. Ensure temporal ordering
df_clean = ensure_temporal_order(df_clean)
print("✓ Temporal order verified")

print(f"\nClean dataset shape: {df_clean.shape}")

## 10. Save Cleaned Dataset (Optional)

In [ ]:
# Uncomment to save cleaned dataset
# output_path = Path('../data/Nfl_data_cleaned.csv')
# df_clean.to_csv(output_path, index=False)
# print(f"Cleaned dataset saved to: {output_path}")

## 11. Run Full Enhanced Pipeline

Execute the complete pipeline including model evaluation.

In [ ]:
# Note: This will rebuild the dataset and evaluate models
# Uncomment to run (may take several minutes)

# results = run_enhanced_pipeline(
#     start_year=2010,
#     end_year=2025,
#     evaluate_models=True,
#     cv_folds=5
# )

# print("\nPipeline results:")
# print(json.dumps(results, indent=2))

## 12. Load and Review Pipeline Reports

After running the pipeline, review the generated reports.

In [ ]:
# Load pipeline summary (if it exists)
reports_dir = Path('../data/reports')

if (reports_dir / 'pipeline_summary.json').exists():
    with open(reports_dir / 'pipeline_summary.json', 'r') as f:
        summary = json.load(f)
    
    print("Pipeline Summary:")
    print(f"  Status: {summary['status']}")
    print(f"  Start: {summary['start_time']}")
    print(f"  End: {summary['end_time']}")
    
    if 'evaluation_results' in summary:
        eval_results = summary['evaluation_results']
        print("\nModel Evaluation Results:")
        print(f"  Home Model R²: {eval_results['home_model']['mean_r2']:.4f} ± {eval_results['home_model']['std_r2']:.4f}")
        print(f"  Away Model R²: {eval_results['away_model']['mean_r2']:.4f} ± {eval_results['away_model']['std_r2']:.4f}")
        print(f"  Win Classifier AUC: {eval_results['win_classifier']['mean_auc']:.4f} ± {eval_results['win_classifier']['std_auc']:.4f}")
else:
    print("No pipeline summary found. Run the enhanced pipeline first.")

## Summary

This notebook demonstrated:

1. ✓ Loading and inspecting NFL dataset
2. ✓ Basic data quality checks
3. ✓ Team code validation and normalization
4. ✓ Temporal ordering verification
5. ✓ Feature validation
6. ✓ Data leakage detection
7. ✓ Enhanced pipeline validation framework
8. ✓ Common data cleaning operations
9. ✓ Full pipeline execution (optional)
10. ✓ Report review and analysis

### Next Steps

- Review any error reports generated during validation
- Apply recovery solutions for any detected issues
- Run the full pipeline to rebuild dataset and evaluate models
- Monitor cross-validation scores for model performance
- Integrate with automated retraining workflow

### Resources

- Enhanced Pipeline Documentation: `docs/report.md`
- Pipeline Source Code: `backend/enhanced_pipeline.py`
- Dataset Builder: `backend/build_csv_datasets.py`
- Model Training: `backend/train_models.py`